In [1]:
import numpy as np
from plotly.io import show

from skfolio.datasets import load_sp500_dataset
from skfolio.distribution import (
    ClaytonCopula,
    Gaussian,
    GaussianCopula,
    GumbelCopula,
    JoeCopula,
    JohnsonSU,
    StudentT,
    StudentTCopula,
    select_bivariate_copula,
    select_univariate_dist,
)
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()
prices = prices[["BAC", "JPM"]]
X = prices_to_returns(prices)
print(X.tail())

                 BAC       JPM
Date                          
2022-12-21  0.015223  0.011248
2022-12-22 -0.008848 -0.011355
2022-12-23  0.002443  0.004749
2022-12-27  0.001875  0.003504
2022-12-28  0.007360  0.005463


In [2]:
#marginal distribution
candidates = [Gaussian(), StudentT(), JohnsonSU()]
X1, X2 = X[["BAC"]], X[["JPM"]]

bac_dist = select_univariate_dist(X=X1, distribution_candidates=candidates)
print(f"BAC: {bac_dist.fitted_repr}")

jpm_dist = select_univariate_dist(X=X2, distribution_candidates=candidates)
print(f"JPM: {jpm_dist.fitted_repr}")

BAC: StudentT(loc=0.00033, scale=0.013, df=2.5)
JPM: JohnsonSU(a=-0.041, b=1.1, loc=-0.00032, scale=0.015)


In [3]:
jpm_dist.plot_pdf(X2)

In [4]:
jpm_dist.qq_plot(X2)

In [5]:
gaussian = Gaussian()
gaussian.fit(X2)
gaussian.plot_pdf(X2)

In [6]:
gaussian.qq_plot(X2)

In [7]:
X = np.hstack([bac_dist.cdf(X1), jpm_dist.cdf(X2)])

In [8]:
#bivariate copulas
candidates = [
    GaussianCopula(),
    StudentTCopula(),
    ClaytonCopula(),
    GumbelCopula(),
    JoeCopula(),
]
copula = select_bivariate_copula(X, copula_candidates=candidates)
print(copula.fitted_repr)
print(f"AIC: {copula.aic(X):,.2f}")

StudentTCopula(rho=0.748, dof=2.65)
AIC: -7,525.40


In [9]:
fig = copula.plot_pdf_2d()
fig.update_layout(height=700)

In [10]:
fig = copula.plot_pdf_3d()
fig.update_layout(scene_camera=dict(eye=dict(x=-1.2, y=1.4, z=0.8)))
fig

In [11]:
print(f"Lower Tail Dependence: {copula.lower_tail_dependence:.2%}")
print(f"Upper Tail Dependence: {copula.upper_tail_dependence:.2%}")

Lower Tail Dependence: 51.21%
Upper Tail Dependence: 51.21%


In [12]:
copula.plot_tail_concentration(X)

In [13]:
copula = GaussianCopula()
copula.fit(X)
print(copula.fitted_repr)
print(f"Rho: {copula.rho_:0.2f}")
print(f"AIC: {copula.aic(X):,.2f}")
print(f"Lower Tail Dependence: {copula.lower_tail_dependence:.2%}")
print(f"Upper Tail Dependence: {copula.upper_tail_dependence:.2%}")

GaussianCopula(rho=0.748)
Rho: 0.75
AIC: -6,346.65
Lower Tail Dependence: 0.00%
Upper Tail Dependence: 0.00%


In [14]:
fig = copula.plot_pdf_2d()
fig.update_layout(height=700)

In [15]:
copula.plot_tail_concentration(X)

In [16]:
copula = JoeCopula()
copula.fit(X)
print(copula.fitted_repr)
print(f"Rotation: {copula.rotation_}")
print(f"Rho: {copula.theta_:0.2f}")
print(f"AIC: {copula.aic(X):,.2f}")
print(f"Lower Tail Dependence: {copula.lower_tail_dependence:.2%}")
print(f"Upper Tail Dependence: {copula.upper_tail_dependence:.2%}")

JoeCopula(theta=3.18, rot=180°)
Rotation: 180°
Rho: 3.18
AIC: -4,921.77
Lower Tail Dependence: 75.61%
Upper Tail Dependence: 0.00%


In [17]:
fig = copula.plot_pdf_2d()
fig.update_layout(height=700)
show(fig)

In [18]:
copula.plot_tail_concentration(X)